# Titanic Exploratory Data Analysis & Data Cleaning

## Project Requirements Covered:
1. **DataFrame operations:** `head()`, `tail()`, `info()`, `describe()`, conditional filtering
2. **Handling missing values and duplicates:** `duplicated()`, `drop_duplicates()`, `isnull()`, imputation strategies
3. **Grouping and aggregation analysis:** `groupby()`, `agg()`, `pd.pivot_table()`, `pd.crosstab()`

**Expected Outcome:** Master Pandas library operations and foundational exploratory data cleaning techniques.

## Part 1: DataFrame Operations (`head`, `info`, `describe`)
In this section, we load the raw dataset, inspect its structure, data types, and review statistical summaries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load dataset
df = pd.read_csv('titanic.csv')

# 2. Inspect first 5 rows
print('--- First 5 Rows (df.head()) ---')
display(df.head())

# 3. Inspect last 3 rows
print('--- Last 3 Rows (df.tail(3)) ---')
display(df.tail(3))

### Interpretation:
- `head()` displays the initial sample rows, revealing column names like `PassengerId`, `Survived`, `Pclass`, `Name`, `Sex`, `Age`, `SibSp`, `Parch`, `Ticket`, `Fare`, `Cabin`, and `Embarked`.
- `tail()` confirms total indexing up to 890 (891 rows total) and shows data structure continuity.

In [ ]:
# 4. Detailed DataFrame Info (Schema, Data Types, Non-Null Counts)
print('--- Schema & Memory Info (df.info()) ---')
df.info()

### Interpretation:
- `info()` reveals 891 entries with a mixture of `int64`, `float64`, and `object` (string) datatypes.
- Notable columns with missing values: `Age` (fewer than 891 non-nulls), `Cabin` (severely sparse), and `Embarked`.

In [ ]:
# 5. Statistical Summaries using describe()
print('--- Numerical Features Summary (df.describe()) ---')
display(df.describe().round(2))

print('--- Categorical Features Summary (df.describe(include=["O"])) ---')
display(df.describe(include=['O']))

### Interpretation:
- **Numerical:** The mean survival rate is ~38.4%. `Fare` has an extreme positive skew (mean = £32.20, median = £14.45, max = £512.33).
- **Categorical:** `Sex` has 2 unique values with `male` being the most frequent (top). `Embarked` has 3 unique values with `S` (Southampton) being the predominant port.

## Part 2: Handling Missing Values and Duplicates
A rigorous data cleaning workflow ensuring zero data loss where possible.

In [ ]:
# 1. Check for Duplicate Rows
num_duplicates = df.duplicated().sum()
print(f'Total Duplicate Rows Found: {num_duplicates}')

if num_duplicates > 0:
    df = df.drop_duplicates()
    print(f'Duplicates dropped. New shape: {df.shape}')
else:
    print('No duplicate rows exist in the dataset.')

In [ ]:
# 2. Identify Missing Values Frequency & Percentages
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_count, 'Percentage (%)': missing_pct.round(2)})
display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))

In [ ]:
# 3. Execute Imputation Pipeline
df_clean = df.copy()

# A. Impute Embarked with the Mode
mode_embarked = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(mode_embarked)

# B. Extract Title and Impute Age with conditional subgroup medians
df_clean['Title'] = df_clean['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df_clean['Title'] = df_clean['Title'].replace(rare_titles, 'Rare')
df_clean['Title'] = df_clean['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Median age grouped by Pclass and Title
age_imputer = df_clean.groupby(['Pclass', 'Title'])['Age'].transform('median')
df_clean['Age'] = df_clean['Age'].fillna(age_imputer)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# C. Handle Cabin: Extract Deck prefix and flag presence
df_clean['Deck'] = df_clean['Cabin'].str[0].fillna('Unknown')
df_clean['HasCabin'] = df_clean['Cabin'].notnull().astype(int)

# 4. Post-Cleaning Verification
print('Remaining nulls after cleaning:')
print(df_clean[['Embarked', 'Age', 'Deck', 'HasCabin']].isnull().sum())

### Interpretation:
- `Embarked` had only 2 missing entries, cleanly resolved via mode (`S`).
- Instead of distorting demographic groups with a blunt global median, `Age` was imputed by `(Pclass, Title)` subgroups (e.g. `Master` children vs `Mr.` adults).
- `Cabin` missingness is informative (Missing Not At Random). Creating `Deck` and `HasCabin` preserves critical socioeconomic information without dropping 77% of records.

## Part 3: Grouping and Aggregation Analysis
Extracting deep domain insights using `groupby()`, `.agg()`, `pivot_table()`, and `crosstab()`.

In [ ]:
# 1. Groupby Single Column: Survival by Sex with multiple metrics
print('--- Survival Metrics by Sex ---')
sex_summary = df_clean.groupby('Sex')['Survived'].agg(
    Total_Passengers='count',
    Survivors='sum',
    Survival_Rate=lambda x: f'{x.mean() * 100:.2f}%'
)
display(sex_summary)

In [ ]:
# 2. Groupby Multi-metric Aggregations across Pclass
print('--- Aggregations by Ticket Class (Pclass) ---')
pclass_metrics = df_clean.groupby('Pclass').agg(
    Passenger_Count=('PassengerId', 'count'),
    Survival_Rate=('Survived', lambda x: f'{x.mean() * 100:.2f}%'),
    Average_Age=('Age', 'mean'),
    Median_Fare=('Fare', 'median'),
    Max_Fare=('Fare', 'max')
).round(2)
display(pclass_metrics)

In [ ]:
# 3. Pivot Table: Interaction of Pclass and Sex on Survival
print('--- Pivot Table: Survival Rate (%) by Pclass and Sex ---')
pivot_table_surv = pd.pivot_table(
    df_clean,
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean',
    margins=True,
    margins_name='Total'
)
display((pivot_table_surv * 100).round(2))

In [ ]:
# 4. Cross-tabulation: Embarkation Port vs Ticket Class
print('--- Cross-tabulation: Embarked vs Pclass ---')
crosstab_res = pd.crosstab(
    df_clean['Embarked'],
    df_clean['Pclass'],
    margins=True,
    margins_name='All'
)
display(crosstab_res)

### Interpretation:
1. **Gender Divide:** Women had a **~74.2%** survival rate compared to **~18.9%** for men, reflecting strict adherence to the maritime code "women and children first".
2. **Class Privilege:** 1st Class passengers experienced a **~63.0%** survival rate versus only **~24.2%** in 3rd Class.
3. **Combined Interaction:** First-class women survived at **~96.8%**, while third-class men suffered an **~86.5% mortality rate**.
4. **Port Distribution:** Most 1st class passengers embarked at Cherbourg ('C') and Southampton ('S'), whereas Queenstown ('Q') passengers were predominantly 3rd class.